# Modül 00 — İnteraktif Keşif

Bu notebook, `.py` dosyalarındaki kavramları **görselle ve denemeyle** pekiştirir. `.py` dosyaları temiz, test edilebilir kodu içerir; notebook ise sezgiyi büyütmek için.

**Önerilen sıra:** her bölümü `01_*.py`, `02_*.py` ... ile paralel oku.

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Grafik stilini sade tutuyoruz — odak veride olsun.
plt.rcParams["figure.figsize"] = (7, 4)
plt.rcParams["axes.grid"] = True

rng = np.random.default_rng(seed=0)

## 1 · Python kısa tekrar — list comprehension hız testi

List comprehension'ın yalnızca okunabilir değil, klasik döngüye göre **daha hızlı** olduğunu doğrulayalım.

In [ ]:
import time

N = 1_000_000

t0 = time.perf_counter()
kareler_dongu = []
for x in range(N):
    kareler_dongu.append(x * x)
t_dongu = time.perf_counter() - t0

t0 = time.perf_counter()
kareler_comp = [x * x for x in range(N)]
t_comp = time.perf_counter() - t0

print(f"for döngüsü     : {t_dongu*1000:.1f} ms")
print(f"comprehension   : {t_comp*1000:.1f} ms")
print(f"NumPy (arange^2): hemen aşağıda...")

In [ ]:
t0 = time.perf_counter()
kareler_np = np.arange(N) ** 2
t_np = time.perf_counter() - t0
print(f"NumPy           : {t_np*1000:.1f} ms")
print("NumPy'nin neden 'standart' olduğunu görüyoruz: vektörel işlem büyük farkla kazanır.")

## 2 · NumPy array'i görselleştir

Bir 2D array'i sayısal değerlere bakmak yerine **renk haritası** olarak görmek, şekil-değer ilişkisini hızla içselleştirmeye yardımcı olur.

In [ ]:
M = np.arange(20).reshape(4, 5)
print("M.shape:", M.shape)
print(M)

plt.imshow(M, cmap="viridis")
plt.colorbar(label="değer")
plt.title("arange(20).reshape(4, 5)")
plt.xlabel("sütun (axis=1)")
plt.ylabel("satır (axis=0)")
plt.show()

In [ ]:
# Dilimleme: ikinci sütunu (axis=1, index=1) kırmızıya boya.
M2 = M.astype(float).copy()
M2[:, 1] = np.nan  # NaN, imshow tarafından boş gösterilir; vurgulu görmek için.

plt.imshow(M2, cmap="viridis")
plt.title("M[:, 1] dilimi — bütün satırların ikinci sütunu")
plt.colorbar()
plt.show()

## 3 · Broadcasting'i görmek

(3,1) bir sütun ile (1,4) bir satır toplandığında (3,4) bir tablo oluşur. Aşağıdaki ısı haritasında bunu net göreceğiz.

In [ ]:
sutun = np.array([[1], [2], [3]])     # (3,1)
satir = np.array([[10, 20, 30, 40]])  # (1,4)
tablo = sutun + satir                  # (3,4)

fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, mat, title in zip(axes,
                          [sutun, satir, tablo],
                          ["sutun (3,1)", "satir (1,4)", "sutun + satir (3,4)"]):
    im = ax.imshow(mat, cmap="plasma", aspect="auto")
    ax.set_title(title)
    for (i, j), v in np.ndenumerate(mat):
        ax.text(j, i, str(int(v)), ha="center", va="center", color="white", fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Klasik broadcast hatası: aynı uzunlukta ama uyumsuz boyutlar.
try:
    np.array([1, 2, 3]) + np.array([1, 2, 3, 4])
except ValueError as e:
    print("Beklenen hata:")
    print(" ", e)

## 4 · Vektör/matris işlemleri — kosinüs benzerliği

100 rastgele 2D vektör üretelim ve hepsinin (1, 0) referansına olan kosinüs benzerliğini görelim.

In [ ]:
V = rng.standard_normal((100, 2))
ref = np.array([1.0, 0.0])

normlar = np.linalg.norm(V, axis=1, keepdims=True)
V_birim = V / normlar

kosinusler = V_birim @ ref  # her satırın ref ile iç çarpımı; ref birim olduğu için doğrudan cos

plt.scatter(V_birim[:, 0], V_birim[:, 1], c=kosinusler, cmap="coolwarm", s=40)
plt.colorbar(label="cos(theta) — ref ile benzerlik")
plt.axhline(0, color="k", lw=0.5)
plt.axvline(0, color="k", lw=0.5)
plt.gca().set_aspect("equal")
plt.title("Birim çember üzerindeki vektörlerin ref=(1,0)'a göre kosinüsü")
plt.show()

## 5 · Eksenler — `axis=0` mı `axis=1` mi?

Bir tabloda axis=0 (sütun bazlı) ve axis=1 (satır bazlı) toplamı görselle ayıralım.

In [ ]:
M = rng.integers(0, 10, size=(4, 5))
axis0 = M.sum(axis=0)  # her sütun için bir toplam → uzunluk 5
axis1 = M.sum(axis=1)  # her satır için bir toplam → uzunluk 4

fig, ax = plt.subplots(figsize=(7, 4))
ax.imshow(M, cmap="viridis")
for (i, j), v in np.ndenumerate(M):
    ax.text(j, i, str(v), ha="center", va="center", color="white")
ax.set_xticks(range(5))
ax.set_yticks(range(4))
ax.set_xticklabels([f"\n↓{s}" for s in axis0])
ax.set_yticklabels([f"→{s}" for s in axis1])
ax.set_title("axis=0 (sütun toplamları altta), axis=1 (satır toplamları sağda)")
plt.show()

In [ ]:
# Sütun bazlı standartlaştırma — döngüsüz.
X = rng.standard_normal((200, 3)) * np.array([1, 10, 0.1]) + np.array([5, -2, 100])
Z = (X - X.mean(axis=0, keepdims=True)) / X.std(axis=0, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, M, t in zip(axes, [X, Z], ["Ham X", "Standartlaştırılmış Z"]):
    for j in range(3):
        ax.hist(M[:, j], bins=30, alpha=0.5, label=f"sütun {j}")
    ax.set_title(t)
    ax.legend()
plt.tight_layout()
plt.show()

## 6 · Rastgelelik — init önemi

Aynı sinir ağı şeklini farklı ölçeklerde başlatınca aktivasyon dağılımı nasıl değişir?

In [ ]:
x = rng.standard_normal((1024, 64))

olcekler = {
    "std=1.0 (çok büyük)": 1.0,
    "std=0.001 (çok küçük)": 0.001,
    "std=1/√fan_in (iyi)": 1.0 / np.sqrt(64),
}

fig, axes = plt.subplots(1, 3, figsize=(12, 3), sharey=True)
for ax, (etiket, s) in zip(axes, olcekler.items()):
    W = rng.standard_normal((64, 64)) * s
    h = x @ W
    ax.hist(h.ravel(), bins=60)
    ax.set_title(f"{etiket}\nstd(h)={h.std():.3f}")
plt.tight_layout()
plt.show()

In [ ]:
# Olasılık dağılımından kategorik örnekleme: dil modeli sampling önizlemesi.
olasilik = np.array([0.7, 0.2, 0.1])
secimler = rng.choice(3, size=10_000, p=olasilik)
frekans = np.bincount(secimler) / 10_000

plt.bar(range(3), olasilik, alpha=0.5, label="p (hedef)")
plt.bar(range(3), frekans, alpha=0.5, label="frekans (gözlem)")
plt.xticks(range(3), ["token 0", "token 1", "token 2"])
plt.legend()
plt.title("10000 örneklemde frekans, hedef olasılığa yakınsıyor")
plt.show()

## 7 · Türev sezgisi — eğimi görmek

f(x) = x² fonksiyonunu çizelim ve x=2 noktasındaki teğeti gösterelim. Sayısal türev = analitik türev mi?

In [ ]:
def sayisal_turev(f, x, h=1e-5):
    return (f(x + h) - f(x - h)) / (2 * h)

f = lambda x: x ** 2
xs = np.linspace(-3, 3, 200)
ys = f(xs)

x0 = 2.0
egim = sayisal_turev(f, x0)
tegen = f(x0) + egim * (xs - x0)

plt.plot(xs, ys, label="f(x) = x²")
plt.plot(xs, tegen, "--", label=f"teğet @ x={x0} (eğim={egim:.3f})")
plt.scatter([x0], [f(x0)], color="red", zorder=3)
plt.axhline(0, color="k", lw=0.5)
plt.axvline(0, color="k", lw=0.5)
plt.legend()
plt.title("Sayısal türev, analitik türevle (2x = 4) çakışıyor")
plt.show()

In [ ]:
# Gradient descent animasyonu (statik): f(x, y) = x² + 3y² için yörünge çizimi.
def grad(v):
    return np.array([2 * v[0], 6 * v[1]])

yol = [np.array([4.0, -3.0])]
lr = 0.1
for _ in range(40):
    yol.append(yol[-1] - lr * grad(yol[-1]))
yol = np.array(yol)

# Eş yükselti eğrileri
gx, gy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
F = gx ** 2 + 3 * gy ** 2

plt.contour(gx, gy, F, levels=20, cmap="viridis")
plt.plot(yol[:, 0], yol[:, 1], "o-", color="red", markersize=4)
plt.scatter([0], [0], color="green", zorder=3, label="minimum")
plt.legend()
plt.gca().set_aspect("equal")
plt.title("f(x, y) = x² + 3y² — gradyan inişi yörüngesi")
plt.xlabel("x"); plt.ylabel("y")
plt.show()

## Modül 00 — özet

Buradan sonra **Modül 01 — Lineer Cebir**'e geçiyoruz. Orada:

- Matris çarpımının geometrik anlamı,
- Özdeğer / özvektör,
- SVD ile boyut indirgeme,

konularını LLM bağlamında işleyeceğiz.

Bu notebook'taki her grafikle ilgili `01_*.py` ... `07_*.py` dosyalarına dönüp **kodu** da incele — orada `if __name__ == "__main__"` blokları ile assert'ler vardır.